# Track 1 Guided Explorer

**For:** learners with no coding experience. 

You will use a prepared NOAA drought snapshot, make one chart, explain one pattern, and name one limitation and one reviewer. You do not need to understand every line of code. Run cells in order with **Shift+Enter**.

**Output status:** classroom draft: do not distribute.

## How to use this notebook

Choose this pathway with your team; the tracks are parallel choices. Run cells from top to bottom. All original analysis and stewardship activities are retained, and supporting Python code is visible here. No `src` folder or custom-module import is needed. Use the prepared offline environment; ask a mentor about unfamiliar functions. Your team develops its own question.

## Learning objectives

By the end of the hackathon, students should be able to:

1. **Frame a relevant question** about climate, agriculture or natural-resource stewardship and explain why it interests them.
2. **Use their selected notebook pathway** to explore environmental data, documenting what they tried and any challenges encountered.
3. **Interpret and communicate evidence**, explaining the source, location, time period and meaning of any results or visualizations they present.
4. **Recognize limitations and uncertainty**, distinguishing what the data show from what would require additional evidence.
5. **Propose a future project**, identifying a next question and the data, skills or partnerships needed to pursue it.
6. **Describe potential community benefits** and explain who might find the work useful.
7. **Identify appropriate reviewers or collaborators** and explain how their perspectives could improve interpretation and guide responsible sharing.

These objectives apply across all three tracks. Students demonstrate learning through their final presentations and explanations of completed or attempted work; a finished visualization is not required.

## Agricultural application

How can regional drought and streamflow records help frame questions about water availability, and what additional local evidence would we need?

Alternative questions may concern grazing, gardens, plant resources or watershed stewardship. State what additional evidence would be needed.

## 1. Frame the question: discuss before coding

Who identified the question? Who could benefit? Who should review the interpretation? Do not enter sensitive or community-held knowledge in this notebook.

## Sovereignty activity 1: who shapes the question? (3-5 minutes)

Data sovereignty concerns Indigenous Peoples' authority over data relationships and uses. Data governance is how decisions about collection, interpretation, access and reuse are put into practice. For this exercise, the question itself is a decision: who chose it, whose priorities does it reflect, and who could change it?

With your group, identify a possible benefit, a role or body whose direction would be needed for a real project, and one kind of information you will **not** collect here. You can discuss a hypothetical situation without sharing personal, cultural or protected knowledge if desired.

Start the decision notes below. Use role descriptions, not private contact details. `None` means unresolved, not permission. Keep sensitive answers outside this notebook in the appropriate setting. See [sovereignty practice](guides/sovereignty_practice.md) for optional framework references.


In [ ]:
# CUSTOMIZE: brief, non-sensitive discussion notes. 
governance_notes = {
    "question": None,
    "potential_benefit": None,
    "authority_to_consult": None,
    "community_input_needed": None,
    "representation_limits": None,
    "reviewer_roles": None,
    "intended_audience": "agreed classroom audience; no public release assumed",
    "storage_and_access": None,
    "reuse_limits": "classroom exercise; revisit purpose and permissions before reuse",
    "correction_withdrawal": None,
    "decision": None,
    "decision_reason": None,
}
print("Start with question, benefit and authority to consult. Leave unknowns as None.")

In [ ]:
# YELLOW: change only the text inside quotation marks.
TEAM_NAME = "Team Name Here"
QUESTION = "How can regional drought and streamflow records help frame questions about livestock water availability, and what additional local evidence would we need?"
INTENDED_REVIEWER = "Name the appropriate community or Tribal reviewer"
print(f"{TEAM_NAME}: {QUESTION}")
print(f"Review before sharing: {INTENDED_REVIEWER}")

## 2. Load the prepared snapshot

This file was downloaded from NOAA and checksummed in `data/sample_or_fallback/manifest.json`. It is a regional climate-division indicator.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

start = Path.cwd().resolve()
REPO = next((p for p in [start, *start.parents] if (p / "data").is_dir()), None)
if REPO is None:
    raise RuntimeError("Repository root not found. Ask a mentor how Jupyter was launched.")
SOURCE = REPO/"data"/"sample_or_fallback"/"noaa_climate_division_pdsi.txt"
if not SOURCE.is_file():
    raise FileNotFoundError("Fallback snapshot missing. Ask the instructor to run prepare_workshop_data.py.")
print(f"READY found {SOURCE.name}")
import json, hashlib
manifest = json.loads((SOURCE.parent / "manifest.json").read_text())
source_entry = next(s for s in manifest["sources"] if s["file"] == SOURCE.name)
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == source_entry["sha256"], "Snapshot checksum mismatch; ask instructor."
print("Snapshot acquired:", source_entry["accessed_utc"])

### Read the drought calculation step by step

The original `annual_pdsi` helper is expanded below. Each cell leaves its tables available for inspection. Run in order: parse monthly values, count valid months, then average complete years. Missing values are never replaced with zero.

In [ ]:
path = SOURCE
start_year = 1980

Parse the NOAA rows: select South Dakota divisions 7 and 8, then keep each month.

In [ ]:
rows = []
for line in Path(path).read_text(encoding='utf-8').splitlines():
    parts = line.split()
    if len(parts) != 13 or len(parts[0]) != 10 or (not parts[0].isdigit()):
        continue
    code = parts[0]
    if code[:2] != '39' or code[4:6] != '05' or int(code[2:4]) not in (7, 8):
        continue
    year = int(code[6:])
    if year < start_year:
        continue
    for month, value in enumerate(parts[1:], 1):
        rows.append(dict(year=year, division=int(code[2:4]), month=month, pdsi=float(value)))

Build the monthly table, reject duplicates, and exclude invalid index values.

In [ ]:
data = pd.DataFrame(rows, columns=['year', 'division', 'month', 'pdsi'])
if data.empty:
    raise ValueError('No matching PDSI records; check source and study period.')
if data.duplicated(['year', 'division', 'month']).any():
    raise ValueError('Duplicate division/month records; inspect source.')
valid = data[data.pdsi.between(-99, 99, inclusive='neither') & data.pdsi.notna()]

Count valid months in both divisions. Keep only complete years, then average the two division means equally.

In [ ]:
years = range(int(data.year.min()), int(data.year.max()) + 1)
counts = valid.groupby(['year', 'division']).size().unstack().reindex(index=years, columns=[7, 8], fill_value=0).fillna(0).astype(int)
audit = counts.rename(columns={7: 'division_7_months', 8: 'division_8_months'})
audit['complete'] = (counts[7] == 12) & (counts[8] == 12)
eligible = audit.index[audit.complete]
annual = valid[valid.year.isin(eligible)].groupby(['year', 'division']).pdsi.mean().groupby('year').mean()
if annual.empty:
    raise ValueError('No complete years for both divisions; select a longer period or inspect snapshot.')

In [ ]:
drought = data
display(drought.head())
display(audit.tail())
display(annual.tail())

In [ ]:
# GREEN: require 12 valid months in each of divisions 7 and 8.
print(f"Complete years: {annual.index.min()}–{annual.index.max()} ({len(annual)})")
print("Excluded years (incomplete division/month coverage):", audit.index[~audit.complete].tolist())
display(audit.tail())

## Sovereignty activity 2: what does this data represent? (3-5 minutes)

The file is a federal regional index. Who chose the division boundaries, and what local differences does the equal-weight summary hide? Compare a cautious description of regional drought with a claim about a specific garden, pasture or household. What locally directed evidence would be needed before making the latter claim? Discuss whether changing the question would be better than adding more data.

Update `representation_limits` and `community_input_needed` below. A data gap is an unanswered question; it does not establish lack of community knowledge.

In [ ]:
# CUSTOMIZE after discussing representation. Use non-sensitive notes only.
governance_notes["representation_limits"] = None
governance_notes["community_input_needed"] = None

## 3. Make and read one chart

Negative PDSI values indicate relatively dry conditions; positive values indicate relatively wet conditions.

Each plotted year has 12 valid months for each division. The annual regional value equally weights the two division means. It is not area-weighted to the reservation. The audit lists excluded years; the current partial year is not silently compared with full years.

In [ ]:
colors = np.where(annual < 0, "#C0392B", "#2471A3")
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(annual.index, annual.values, color=colors, width=0.85)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Regional drought conditions for South Dakota climate divisions 7 and 8", xlabel="Year", ylabel="Equal-weight annual mean PDSI")
ax.text(0, -0.20, "NOAA/NCEI PDSI; complete years only; equal division weights; classroom draft; regional indicator", transform=ax.transAxes, fontsize=8)
plt.tight_layout()
plt.show()

## 4. Interpret before calculating more

Complete these in your own words:

- **Pattern:** The chart shows...
- **Evidence:** I based that on...
- **Limitation:** This chart cannot tell us...
- **Community check:** A locally meaningful interpretation would also need...
- **Reviewer:** Before sharing, this should be reviewed by...

In [ ]:
# YELLOW: replace the prompts with your team's words.
FINDING = "We observed..."
EVIDENCE = "The chart shows..."
LIMITATION = "This regional indicator cannot tell us..."
print(FINDING)
print(EVIDENCE)
print(LIMITATION)
print("Status: classroom draft: do not distribute")

Once you feel comfortable running this notebook, explore the other repositories on the OLC GitHub. If you have a specific question to ask and are not sure where to begin, ask a mentor to help get you started. Once you feel comfortable running this notebook, explore the other repositories on the OLC GitHub. If you have a specific question to ask and are not sure where to begin, ask a mentor to help get you started. 

## Sovereignty activity 3: choose the next use (5 minutes)
Imagine a request to post your chart online or reuse it in a project about a different community. What changes in purpose, audience or representation would need review? Discuss an option to revise, limit or decline the proposed use. Who can ask for a correction or withdrawal, and who would act on it?

Record your decision and reason, potential benefit, reviewer roles, storage/access and correction process. You may leave unresolved items as `None` and explain what needs to happen next. Keep real sensitive information out of this exercise.


In [ ]:
# CUSTOMIZE: a proposed classroom decision, not an authorization.
governance_notes.update({
    "potential_benefit": None,
    "authority_to_consult": None,
    "reviewer_roles": None,
    "storage_and_access": None,
    "correction_withdrawal": None,
    "decision": None,  # ex., revise the claim; keep within class; seek review before reuse
    "decision_reason": None,
})

## Provenance: technical lineage and governance decisions

The cell records the actual snapshot URLs/dates and verified checksums, notebook identity, analytical settings, limitations and your discussion notes. Review these together.

The [IEEE 2890-2025 recommended practice](https://standards.ieee.org/ieee/2890/10318/) addresses provenance relevant to Indigenous Peoples' data relationships and governance. This classroom record explores those ideas; it is not a standards-conformity assessment or evidence of Tribal endorsement. The [framework references](guides/sovereignty_practice.md) provide further reading.


### Visible provenance code

A function is a named set of steps. `def` defines it; the later call runs it with our files and discussion notes. All of the code is here. The first function verifies sources and builds a record; the second saves a local draft only when requested. A checksum identifies file contents; it does not grant permission to share.

In [ ]:
"""Local teaching provenance; records evidence without granting permissions."""

In [ ]:
from copy import deepcopy

In [ ]:
from datetime import datetime, timezone

In [ ]:
import hashlib

In [ ]:
import json

In [ ]:
from pathlib import Path

In [ ]:
import platform

In [ ]:
GOVERNANCE_FIELDS = ('question','potential_benefit','authority_to_consult','community_input_needed',
                     'representation_limits','reviewer_roles','intended_audience','storage_and_access',
                     'reuse_limits','correction_withdrawal','decision','decision_reason')

Verify snapshot checksums, record the notebook and analysis settings, and list unresolved discussion fields.

In [ ]:
def build_provenance(root, notebook, source_files, analysis, governance):
    root=Path(root).resolve()
    snapshot=root/'data/sample_or_fallback'
    manifest=json.loads((snapshot/'manifest.json').read_text(encoding='utf-8'))
    indexed={item['file']:item for item in manifest['sources']}
    sources=[]
    for name in dict.fromkeys(source_files):
        path=(snapshot/name).resolve()
        if not path.is_relative_to(snapshot.resolve()):raise ValueError('Source path escapes snapshot folder.')
        if name not in indexed:raise ValueError(f'No source manifest entry for {name}')
        actual=hashlib.sha256(path.read_bytes()).hexdigest()
        if actual!=indexed[name]['sha256']:raise ValueError(f'Source checksum mismatch: {name}')
        item=deepcopy(indexed[name])
        item['checksum_verified']=True
        sources.append(item)
    if not sources:raise ValueError('At least one source is required.')
    missing=[key for key in GOVERNANCE_FIELDS if not isinstance(governance.get(key),str) or not governance[key].strip()]
    nb_path=(root/notebook).resolve()
    if not nb_path.is_relative_to(root):raise ValueError('Notebook must be in the repository.')
    record=dict(schema_version='1.0',created_utc=datetime.now(timezone.utc).isoformat(),
        status='classroom draft: external sharing not approved by this record',
        technical_lineage=dict(notebook=notebook,notebook_sha256=hashlib.sha256(nb_path.read_bytes()).hexdigest(),
            implementation_location='visible notebook cells',python=platform.python_version(),
            sources=sources,analysis=deepcopy(analysis)),
        governance_decisions=deepcopy(governance),unresolved_fields=missing,
        evidence_note='Student discussion record; reviewer roles are proposed, not confirmed authorization.',
        standards_note='Teaching record informed by provenance concepts; no IEEE 2890 conformity assessment performed.')
    # Check serialization now, so failure happens before a student chooses to save.
    json.dumps(record,allow_nan=False)
    return record

Save a uniquely named JSON file without overwriting another draft. The optional save cell below controls whether this runs.

In [ ]:
def save_draft(record, folder):
    if not record.get('status','').startswith('classroom draft:'):
        raise ValueError('This helper saves classroom drafts only; it cannot grant approval.')
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    path=folder/('provenance-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'.json')
    with path.open('x',encoding='utf-8') as stream:
        json.dump(record,stream,indent=2,allow_nan=False);stream.write('\n')
    return path

In [ ]:
import json
source_files = [SOURCE.name]
analysis = {
    "source_steward": "NOAA/NCEI",
    "measure": "equal-weight annual mean PDSI for South Dakota divisions 7 and 8",
    "units": "PDSI index (dimensionless)",
    "period": [int(annual.index.min()), int(annual.index.max())],
    "included_years": [int(y) for y in annual.index],
    "excluded_years": [int(y) for y in audit.index[~audit.complete]],
    "completeness_rule": "12 valid months in each division; both divisions required",
    "processing": ["parse fixed-width monthly rows", "exclude incomplete years", "average each division, then equally weight division means"],
    "limitations": ["regional divisions are not a reservation measurement", "equal weights are not area weights", "no direct measure of livestock supply, water quality or garden conditions"],
    "artifact": "in-notebook regional drought chart; no external publication",
}
provenance = build_provenance(REPO, "01_guided_explorer.ipynb", source_files, analysis, governance_notes)
print(json.dumps(provenance, indent=2))
print("Unresolved discussion fields:", provenance["unresolved_fields"])
# Completing fields does not change the classroom-draft status.

### Optional: save a local draft record

If your notes contain only appropriate classroom information, set `SAVE_DRAFT=True` to save a JSON sidecar under ignored `outputs/governance/`. It describes the in-notebook artifacts; it does not export a figure or publish anything. Use the agreed class storage for completed work. The notebook itself can also retain edited notes but clear private information before sharing the notebook. Re-run the provenance cell after changing analysis settings or notes.


In [ ]:
SAVE_DRAFT = False
if SAVE_DRAFT:
    draft_path = save_draft(provenance, REPO/"outputs"/"governance")
    print("Saved local classroom draft:", draft_path)
else:
    print("Draft displayed above; no sidecar saved.")

## What to bring to the final presentation

Explain what you tried and learned; show any visualizations and describe their source and meaning. Record uncertainty and future project ideas. Use the seven questions below to prepare.

## Final presentation and stewardship

1. **What question did you explore, and why did it interest you?**
2. **What did you learn?** Describe something about the topic, data or method.
3. **What did you create?** Show any visualizations and explain their source, place, period and meaning. If you did not finish a visualization, explain what you tried and what happened.
4. **What remains uncertain?** Identify a limitation, challenge or unanswered question.
5. **What would you investigate next?** Suggest a future project and the data, skills or partnerships it would need.
6. **Who might benefit from this work?** Explain the potential benefit without claiming an outcome the project has not demonstrated.
7. **Who should review or help interpret this work?** Identify relevant people or roles and why their perspective matters.

See the [presentation guidance](guides/final_presentation.md). Save full stewardship details in the agreed class location. The final hour, September 16 10:45–11:45, is for student sharing and questions.

### Bring your decisions into the seven-question presentation

Use the provenance record to explain sources and limitations (questions 3-4). Explain a future-use decision (question 5), a possible benefit (question 6) and the reviewer roles/perspectives needed (question 7). You can state that a decision remains unresolved. 
